# GROMACS User Guide

**Version:** GROMACS 2026.1

**Audience:** Scientists and researchers running molecular dynamics simulations and trajectory analysis on HPC systems

## TL;DR

This guide explains **how to use GROMACS effectively on this HPC system**.

**What is GROMACS?** GROMACS is a free and open-source software suite for high-performance molecular dynamics and output analysis.

Run one simple case first:

```bash
# 1. Load the module
module load gromacs

# 2. Verify install
gmx --version

# 3. Run a minimal command
gmx help
```

It emphasizes:
- Quick setup for first-time users
- Common usage patterns and workflows
- Troubleshooting and getting help

The goal is to **get you from zero to your first result in 15 minutes**, then move into production workflows.

## Table of Contents

- [TL;DR](#tldr)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute QuickStart](#tldr-5-minute-quickstart)
  - [Step 1: Load the Module](#step-1-load-the-module)
  - [Step 2: Verify Setup](#step-2-verify-setup)
  - [Step 3: Run Your First Job](#step-3-run-your-first-job)
- [PART II — How to Use GROMACS (Read Carefully Once)](#part-ii--how-to-use-gromacs-read-carefully-once)
  - [What You Need to Know](#what-you-need-to-know)
  - [Core `gmx` Commands and Options](#core-gmx-commands-and-options)
  - [Resource Requests](#resource-requests)
- [PART III — Workflows (Use As Needed)](#part-iii--workflows-use-as-needed)
  - [Workflow 1: CPU Batch Simulation on Build Partition](#workflow-1-cpu-batch-simulation-on-build-partition)
  - [Workflow 2: Prepare a System for MD](#workflow-2-prepare-a-system-for-md)
  - [Workflow 3: Basic Trajectory Analysis](#workflow-3-basic-trajectory-analysis)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Symptom-Based Lookup](#symptom-based-lookup)
  - [FAQ](#faq)
  - [External Resources](#external-resources)

---

## PART I — Getting Started (Read Once)

**Purpose:** "I just want this to work."

This part should feel **quick and confidence-building**. By the end, you will have a working setup and have run your first command successfully. You do not need to understand everything yet, just follow the steps.

### TL;DR: 5-Minute QuickStart

If you just want to run a command:

```bash
# 1. Load the module
module load gromacs

# 2. Verify it worked
gmx --version

# 3. Run a basic command
gmx help
```

**Expected result:** You should see the GROMACS version and command help text.

---

### Step 1: Load the Module

All software on this system is pre-installed and managed through the **module system** (lmod). To get started, load GROMACS:

In [ ]:
%%bash
module avail gromacs 2>&1 | head -30

**Expected output:**
```
...
gromacs
gromacs/2026.1
...
```

Now load the module:

In [ ]:
%%bash
module load gromacs
module list 2>&1 | head -20

**Expected output:**
```
Currently Loaded Modules:
  1) gromacs
```

---

### Step 2: Verify Setup

Confirm the module loaded correctly and your environment is ready:

In [ ]:
%%bash
module load gromacs
gmx --version 2>&1 | head -25

**Expected output:**
```
GROMACS version:    2026.1
...
```

✅ **Success!** Your environment is ready to use.

---

### Step 3: Run Your First Job

Here is the simplest possible workflow to confirm everything works end-to-end:

In [ ]:
%%bash
module load gromacs
gmx help 2>&1 | head -40

**Expected output:**
```
GROMACS:    gmx help [COMMAND]

The following commands are available:
  grompp
  mdrun
  trjconv
  energy
...
```

🎉 **You are set up!** Continue to Part II to understand command choices, or jump to Part III to run common workflows.

**Next Steps:**
- **Need command strategy?** Continue to [Part II](#part-ii--how-to-use-gromacs-read-carefully-once).
- **Ready for practical runs?** Jump to [Part III](#part-iii--workflows-use-as-needed).
- **Hit an error?** Go to [Part IV](#part-iv--troubleshooting-skim-when-broken).

---

## PART II — How to Use GROMACS (Read Carefully Once)

**Purpose:** "Help me not screw this up."

This part explains key concepts and decision-making. You will understand **when** to use different commands and **why** certain choices matter.

### What You Need to Know

A typical GROMACS run has three phases:
1. Build topology and coordinates (`gmx pdb2gmx`, `gmx editconf`, `gmx solvate`)
2. Prepare run input (`gmx grompp`)
3. Execute simulation (`gmx mdrun`) and analyze (`gmx energy`, `gmx trjconv`, `gmx rms`)

#### Supported Environments

| Platform | Status | Notes |
|----------|--------|-------|
| CPU-only | ✅ Full support | Great for teaching, validation, and moderate jobs |
| GPU | ✅ Full support | Faster for large production simulations |
| Login node | ⚠️ Limited | Use only for quick setup/inspection commands |

### Core `gmx` Commands and Options

The top-level `gmx` command supports these common global options:
- `-[no]h`: print help and quit
- `-[no]quiet`: reduce startup output
- `-[no]version`: print version information and quit
- `-[no]copyright`: print copyright information
- `-nice <int>`: set process nice level
- `-[no]backup`: control backup behavior for existing outputs

To inspect command categories and options directly:

In [ ]:
%%bash
module load gromacs
gmx --help 2>&1 | head -80

**Expected output:**
```
gmx [-[no]h] [-[no]quiet] [-[no]version] ...
Description: molecular dynamics simulation suite
...
```

### Resource Requests

- **Quick command checks and tiny preprocessing:** No allocation needed; run on login node.
- **Compute-intensive MD:** Use a SLURM batch job on a compute partition.
- **For this guide:** Workflow 1 uses a **build partition CPU job**.

⚠️ Do not run long `mdrun` jobs on login nodes.

---

## PART III — Workflows (Use As Needed)

**Purpose:** "Show me how to actually do science."

Pick the workflow that matches your task. Each is self-contained.

### Workflow 1: CPU Batch Simulation on Build Partition

Create an example SLURM script for a CPU simulation run:

In [ ]:
%%bash
module load gromacs
cat > gromacs_build_cpu.sbatch << 'EOF'
#!/bin/bash
#SBATCH -J gmx_cpu
#SBATCH -p build
#SBATCH -N 1
#SBATCH -n 32
#SBATCH --mem=64G
#SBATCH -t 08:00:00
#SBATCH -o gmx_cpu_%j.out
#SBATCH -e gmx_cpu_%j.err

module load gromacs

# 1) Build portable run input from topology + mdp
gmx grompp -f md.mdp -c conf.gro -p topol.top -o md.tpr

# 2) Run CPU simulation
gmx mdrun -s md.tpr -deffnm md_run
EOF

sed -n '1,40p' gromacs_build_cpu.sbatch

**Expected output:**
```
#!/bin/bash
#SBATCH -p build
#SBATCH -n 32
...
gmx grompp -f md.mdp -c conf.gro -p topol.top -o md.tpr
gmx mdrun -s md.tpr -deffnm md_run
```

Submit and track it:

In [ ]:
%%bash
module load gromacs
sbatch gromacs_build_cpu.sbatch
squeue -u "$USER" | head -20

**Expected output:**
```
Submitted batch job 123456
JOBID PARTITION NAME    USER ST TIME NODES NODELIST(REASON)
123456 build     gmx_cpu user R  0:01 1     ...
```

### Workflow 2: Prepare a System for MD

This workflow prepares an input system from a structure file:

In [ ]:
%%bash
module load gromacs
cat << 'EOF' > prep_commands.sh
gmx pdb2gmx -f protein.pdb -o processed.gro -water tip3p
gmx editconf -f processed.gro -o boxed.gro -c -d 1.0 -bt cubic
gmx solvate -cp boxed.gro -cs spc216.gro -o solvated.gro -p topol.top
EOF

sed -n '1,20p' prep_commands.sh

**Expected output:**
```
gmx pdb2gmx -f protein.pdb -o processed.gro -water tip3p
gmx editconf -f processed.gro -o boxed.gro -c -d 1.0 -bt cubic
gmx solvate -cp boxed.gro -cs spc216.gro -o solvated.gro -p topol.top
```

### Workflow 3: Basic Trajectory Analysis

After a run, extract common analysis metrics:

In [ ]:
%%bash
module load gromacs
cat << 'EOF' > analysis_commands.sh
gmx energy -f md_run.edr -o potential.xvg
gmx rms -s md_run.tpr -f md_run.xtc -o rmsd.xvg
gmx trjconv -s md_run.tpr -f md_run.xtc -o md_run_nojump.xtc -pbc nojump
EOF

sed -n '1,20p' analysis_commands.sh

**Expected output:**
```
gmx energy -f md_run.edr -o potential.xvg
gmx rms -s md_run.tpr -f md_run.xtc -o rmsd.xvg
gmx trjconv -s md_run.tpr -f md_run.xtc -o md_run_nojump.xtc -pbc nojump
```

---

## PART IV — Troubleshooting (Skim When Broken)

**Purpose:** "Something failed. Give me answers fast."

### Symptom-Based Lookup

| Symptom | Likely Cause | Fix |
|---------|--------------|-----|
| `gmx: command not found` | Module not loaded | Run `module load gromacs` |
| `There were X warnings in grompp` | Topology/parameter mismatch | Review `grompp` warnings and fix inputs; avoid blindly using `-maxwarn` |
| Job pending in SLURM | Requested resources unavailable | Reduce cores/memory/time or wait for build partition availability |
| `Fatal error` in `mdrun` after startup | Corrupt/incompatible input files | Re-run `gmx grompp` and validate `.mdp`, topology, and coordinates |
| Job killed by scheduler | Memory/time too low | Increase `#SBATCH --mem` or walltime and resubmit |

### FAQ

**Q: Should I run `gmx mdrun` directly on a login node?**

A: No for production. Use SLURM batch jobs on compute partitions for any meaningful simulation.

**Q: How do I check whether my job is still running?**

A: Use `squeue -u $USER`, then inspect `gmx_cpu_<jobid>.out` and `gmx_cpu_<jobid>.err`.

**Q: What command gives me the full list of GROMACS tools?**

A: Run `gmx help` or `gmx --help` to see command categories and available programs.

### External Resources

- Official GROMACS home page: https://www.gromacs.org/
- Command-line reference (`gmx`): https://manual.gromacs.org/current/onlinehelp/gmx.html
- Full current manual: https://manual.gromacs.org/current/index.html
- Release notes and downloads: https://manual.gromacs.org/current/download.html
- Tutorials and webinars: https://www.gromacs.org/tutorial_webinar.html